# Names

Date: 2024/01/21, 2024/04/19(SQLite), 2025/07/12(Gemini replacing spaCy)

In [1]:
#!pip3 install google-genai
#!pip3 install pandas

In [2]:

import google.genai as genai
import os

GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
MODEL = "gemini-2.5-flash"

client = genai.Client(api_key=GEMINI_API_KEY)

## Preprocessing and Paragraph-Level Splitting of the Original Text

In [3]:
import sqlite3

import pandas as pd
with sqlite3.connect('../data/bach.db') as conn:
    paragraphs = pd.read_sql('SELECT * FROM paragraphs', conn)
    
paragraphs.head()

,index,paragraph
0,0,If there is such a thing as inherited aptitude...
1,1,"Veit Bach, ancestor of this famous family, gai..."
2,2,"Not all the Bachs, however, were great musicia..."
3,3,We do not know whether they rewarded the expec...
4,4,"Besides these three men, the Bachs boasted sev..."


In [4]:
len(paragraphs)

159

In [5]:
response = client.models.generate_content(
    model=MODEL,
    contents=f"""
    Extract the names of all persons mentioned in the following paragraphs.
    If a name is mentioned multiple times, include it only once.
    If a name is misspelled, correct it.
    If a name is not full name, make it full name.

    ## Paragraphs     
    {paragraphs.paragraph.to_list()}
    """,
    config={
        "response_mime_type": "application/json",
        "response_schema": list[str],
    }
)

names = response.parsed
names

['Veit Bach',
 'Johann Sebastian Bach',
 'Georg Frideric Handel',
 'Zachau',
 'Kirchhoff',
 'Louis Marchand',
 'Jean-Baptiste Volumier',
 'Count Flemming',
 'Leopold, Prince of Anhalt-Köthen',
 'Johann Kuhnau',
 'Carl Philipp Emanuel Bach',
 'Frederick II of Prussia',
 'Wilhelm Friedemann Bach',
 'Johann Ambrosius Bach',
 'Johann Christoph Bach',
 'Johann Jacob Froberger',
 'Johann Caspar Ferdinand Fischer',
 'Johann Caspar Kerl',
 'Johann Pachelbel',
 'Dietrich Buxtehude',
 'Nicolaus Bruhns',
 'Georg Böhm',
 'Erdmann',
 'Johann Adam Reinken',
 'Anna Magdalena Bach',
 'Regine Susanna Bach',
 'Ludwig van Beethoven',
 'Johann Christoph Friedrich Bach',
 'Johann Christian Hoffmann',
 'Marianne von Ziegler',
 'Johann Christoph Gottsched',
 'Johann Abraham Birnbaum',
 'Christian Friedrich Henrici',
 'Christian Weiss',
 'Johann Christoph Altnikol',
 'Elias Gottlieb Haussmann',
 'Johann Christian Kittel',
 'Fritz Volbach',
 'Antonio Vivaldi',
 'Reinhard Keiser',
 'Angelo Berardi',
 'Giovanni 

## Export data

In [6]:
import sqlite3
import pandas as pd

names = pd.Series(names, name='name')
with sqlite3.connect('../data/bach.db') as conn:
    names.to_sql('cities', conn, if_exists='replace')    